# 🛒 E-Commerce Data Analysis Project (Olist Dataset)

## 📌 1. فهم هيكل البيانات والروابط (Data Schema)

في هذه المرحلة، نقوم باستكشاف الجداول الأساسية وفهم العلاقات بينها لتسهيل عملية دمج البيانات (`pd.merge`) لاحقاً.

### 🏢 محور العملاء والطلبات (Customers & Orders)

| اسم الجدول | الوصف | الأعمدة الأساسية (Columns) |
| :--- | :--- | :--- |
| `orders` | **قلب المشروع:** يمثل رحلة كل شحنة وحالتها وتواريخها. | `order_id` (مفتاح أساسي), `customer_id`, `order_status`, تواريخ الشحن والتوصيل (`order_purchase_timestamp`, إلخ). |
| `customers` | يحتوي على تفاصيل العملاء وأماكنهم الجغرافية. | `customer_id` (مربوط بالطلب), `customer_unique_id` (المعرف الثابت للعميل)، والمدينة والولاية. |

### 📦 محور المنتجات والبائعين (Products & Sellers)

| اسم الجدول | الوصف | الأعمدة الأساسية (Columns) |
| :--- | :--- | :--- |
| `order_items` | يفك تفاصيل كل طلب (المنتجات والبائعين داخل الطلب الواحد). | `order_id`, `product_id`, `seller_id`, `price` (السعر), `freight_value` (مصاريف الشحن). |
| `products` | مواصفات المنتجات وأبعادها وفئاتها. | `product_id`, `product_category_name` (الفئة بالبرتغالي), أوزان وأبعاد المنتج. |
| `sellers` | تفاصيل البائعين الذين يعرضون المنتجات. | `seller_id`, المدينة والولاية الخاصة بالبائع. |
| `category_translation`| قاموس لترجمة فئات المنتجات. | `product_category_name`, `product_category_name_english`. |

### 💳 محور المعاملات والتقييمات (Payments & Reviews)

| اسم الجدول | الوصف | الأعمدة الأساسية (Columns) |
| :--- | :--- | :--- |
| `order_payments` | طرق الدفع المتنوعة للطلبات والأقساط. | `order_id`, `payment_type` (نوع الدفع), `payment_installments` (الأقساط), `payment_value`. |
| `order_reviews` | تقييمات العملاء وآرائهم بعد استلام الشحنة. | `review_id`, `order_id`, `review_score` (التقييم من 1 لـ 5). |

الكود اللي هتكتبه في أول خلية (Cell) عندك عشان تستدعي المكتبات وتحمل كل الجداول:

In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# 1. تحميل كافة الجداول
customers = pd.read_csv('olist_customers_dataset.csv')
geolocation = pd.read_csv('olist_geolocation_dataset.csv')
order_items = pd.read_csv('olist_order_items_dataset.csv')
order_payments = pd.read_csv('olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('olist_order_reviews_dataset.csv')
orders = pd.read_csv('olist_orders_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
sellers = pd.read_csv('olist_sellers_dataset.csv')
category_translation = pd.read_csv('product_category_name_translation.csv')

print("تم تحميل جميع الملفات بنجاح!")

تم تحميل جميع الملفات بنجاح!


## ⚙️ 2. استكشاف البيانات المبدئي (Initial Exploration)

الآن سنقوم بتشغيل كود لمعرفة حجم الجداول، والتحقق من وجود قيم مفقودة (**Missing Values**)، والتأكد من أنواع البيانات (**Data Types**) وخاصة التواريخ لنبدأ مرحلة التنظيف.

In [4]:
    # كود سريع لاستكشاف حجم الجداول والـ Missing values في جدول الطلبات كمثال
print(f"عدد صفوف جدول الطلبات: {orders.shape[0]}")
print("\nالقيم المفقودة في جدول الطلبات:")
print(orders.isnull().sum())

print("\nأنواع البيانات في جدول الطلبات:")
print(orders.dtypes)

عدد صفوف جدول الطلبات: 99441

القيم المفقودة في جدول الطلبات:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

أنواع البيانات في جدول الطلبات:
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object


## 🧹 3. تنظيف البيانات وتجهيزها (Data Cleaning & Preprocessing)

في هذه الخطوة، نقوم بمعالجة المشاكل الأساسية في البيانات لضمان دقة التحليل:
1. **تحويل التواريخ:** أعمدة التواريخ تم قراءتها كنصوص (`object`)، نقوم بتحويلها إلى صيغة `datetime` لنتمكن من عمل تحليل زمني (Time-Series Analysis).
2. **التعامل مع القيم المفقودة:** نحدد القيم الناقصة ونفهم سببها (مثل عدم وجود تاريخ توصيل للطلبات التي ألغيت أو لم تشحن بعد).

In [5]:
# 1. تحويل أعمدة التواريخ في جدول الطلبات إلى Datetime صحيحة
date_columns = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

# 2. معالجة القيم المفقودة (Missing Values)
# التواريخ الناقصة في وقت التوصيل الفعلي تعني غالباً أن الطلب لم يُسلّم بعد (ملغي أو جاري الشحن)
# سنقوم بملء القيم المفقودة في أسماء الفئات بكلمة 'unknown' لحين دمج الجداول
products['product_category_name'] = products['product_category_name'].fillna('unknown')

print("تم تنظيف التواريخ ومعالجة القيم المفقودة الأساسية!")

تم تنظيف التواريخ ومعالجة القيم المفقودة الأساسية!


## 🔗 4. دمج الجداول وبناء قاعدة التحليل (Data Merging)

بناءً على مخطط العلاقات (ERD)، سنقوم بدمج الجداول لإنشاء مجموعات بيانات متكاملة تسهل عملية التحليل:
1. **جدول المبيعات الرئيسي (`df_sales`):** يجمع بين تفاصيل الطلب، العميل، المنتجات، وأسماء الفئات بالإنجليزية.
2. **جدول المدفوعات والتقييمات:** لربط طرق الدفع والتقييمات بالطلبات.

In [6]:
# أ. دمج جدول المنتجات مع جدول الترجمة للحصول على الأسماء بالإنجليزية
products_translated = pd.merge(
    products, 
    category_translation, 
    on='product_category_name', 
    how='left'
)
# ملء الفئات التي لم تترجم بكلمة 'other'
products_translated['product_category_name_english'] = products_translated['product_category_name_english'].fillna('other')

# ب. بناء الجدول الرئيسي للمبيعات (دمج الطلبات + تفاصيل المنتجات + العملاء + المنتجات المترجمة)
# نربط الطلبات بتفاصيل المنتجات أولاً
df_sales = pd.merge(orders, order_items, on='order_id', how='inner')

# نربط الناتج ببيانات العملاء
df_sales = pd.merge(df_sales, customers, on='customer_id', how='inner')

# نربط الناتج بمواصفات المنتجات المترجمة
df_sales = pd.merge(df_sales, products_translated, on='product_id', how='left')

# ج. دمج البائعين لمعرفة تفاصيلهم الجغرافية
df_sales = pd.merge(df_sales, sellers, on='seller_id', how='left')

print(f"تم إنشاء الجدول الرئيسي بنجاح! حجم البيانات الآن: {df_sales.shape}")
print("الأعمدة المتاحة في الجدول الرئيسي للتحليل:")
print(df_sales.columns.tolist())

تم إنشاء الجدول الرئيسي بنجاح! حجم البيانات الآن: (112650, 30)
الأعمدة المتاحة في الجدول الرئيسي للتحليل:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


## 📊 5. التحليل المالي واستكشاف المبيعات (Sales & Revenue Analysis)

الآن بعد أن أصبحت البيانات جاهزة ومدمجة، سنبدأ بالإجابة على السؤال الأول: **ما هو تطور إجمالي المبيعات (Revenue) على مدار الأشهر؟**
سنقوم باستخراج السنة والشهر من تاريخ الشراء، ثم نستخدم مكتبة **Plotly** لرسم خط بياني تفاعلي (Line Chart).


In [8]:
# 1. استخراج السنة والشهر من تاريخ الشراء
df_sales['year_month'] = df_sales['order_purchase_timestamp'].dt.to_period('M').astype(str)

# 2. حساب إجمالي المبيعات لكل شهر (السعر * العدد)
# بما أن كل سطر يمثل قطعة، فإجمالي المبيعات هو مجموع عمود price
monthly_sales = df_sales.groupby('year_month')['price'].sum().reset_index()

# 3. رسم الرسم البياني التفاعلي باستخدام Plotly Express
fig = px.line(
    monthly_sales, 
    x='year_month', 
    y='price',
    title='تطور إجمالي المبيعات الشهري (Olist Revenue Trend)',
    labels={'year_month': 'الشهر والسنة', 'price': 'إجمالي المبيعات (BRL)'},
    markers=True # لإضافة نقاط عند كل شهر
)

# تحسين مظهر الرسم البياني
fig.update_layout(title_x=0.5, template='plotly_dark')
fig.show()

## 🗺️ 6. التحليل الجغرافي وتوزيع العملاء (Geographical Analysis)

في هذا الجزء، سنقوم باستكشاف التوزيع الجغرافي للعملاء لمعرفة أقوى الولايات بيعاً. 
سنقوم بحساب عدد الطلبات وإجمالي المبيعات لكل ولاية (`customer_state`)، ثم نستخدم **Plotly Express** لرسم مخطط تفاعلي يوضح الترتيب.

In [9]:
# 1. تجميع البيانات حسب الولاية لحساب إجمالي المبيعات وعدد الطلبات الفريدة
state_analysis = df_sales.groupby('customer_state').agg(
    total_revenue=('price', 'sum'),
    order_count=('order_id', 'nunique')
).reset_index()

# ترتيب الولايات من الأعلى للأقل مبيعاً
state_analysis = state_analysis.sort_values(by='total_revenue', ascending=False)

# 2. رسم Bar Chart تفاعلي باستخدام Plotly لـ إجمالي المبيعات حسب الولاية
fig_state = px.bar(
    state_analysis, 
    x='customer_state', 
    y='total_revenue',
    title='إجمالي المبيعات حسب ولاية العميل (Revenue per Customer State)',
    labels={'customer_state': 'الولاية', 'total_revenue': 'إجمالي المبيعات (BRL)'},
    text_auto='.2s', # إظهار الأرقام فوق الأعمدة بشكل مختصر ذكي
    color='total_revenue', # تلوين الأعمدة حسب حجم المبيعات
    color_continuous_scale='Viridis'
)

# تحسين المظهر
fig_state.update_layout(title_x=0.5, template='plotly_dark', xaxis_tickangle=-45)
fig_state.show()

In [10]:
# تجميع داتا الجيو لوكيشن لأخذ متوسط الإحداثيات لكل رمز بريدي (عشان نسرع الرسم)
geo_clean = geolocation.groupby('geolocation_zip_code_prefix').agg(
    lat=('geolocation_lat', 'mean'),
    lng=('geolocation_lng', 'mean')
).reset_index()

# دمج الإحداثيات مع جدول العملاء الرئيسي
customer_geo = pd.merge(
    customers, 
    geo_clean, 
    left_on='customer_zip_code_prefix', 
    right_on='geolocation_zip_code_prefix', 
    how='inner'
)

# عينة عشوائية مكونة من 5000 عميل فقط لسرعة الرسم على الخريطة
geo_sample = customer_geo.sample(n=5000, random_state=42)

# رسم الخريطة التفاعلية
fig_map = px.scatter_mapbox(
    geo_sample, 
    lat="lat", 
    lon="lng", 
    color="customer_state", 
    hover_name="customer_city",
    zoom=3, 
    height=600,
    title='خريطة تفاعلية لتوزيع العملاء في البرازيل'
)

# استخدام ستايل خريطة مفتوح ومجاني
fig_map.update_layout(
    mapbox_style="open-street-map", 
    title_x=0.5,
    margin={"r":0,"t":40,"l":0,"b":0},
    template='plotly_dark'
)
fig_map.show()

C:\Users\MOhamed\AppData\Local\Temp\ipykernel_720\4241874287.py:20: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_map = px.scatter_mapbox(


## 📦 7. تحليل فئات المنتجات (Product Category Analysis)

في هذا الجزء، سنقوم بتحليل المنتجات لمعرفة:
1. **أعلى الفئات مبيعاً من حيث عدد القطع المطلوبة.**
2. **أعلى الفئات تحقيقاً للإيرادات (Revenue).**

سنقوم باستخدام الـ **Treemap** من مكتبة Plotly لتمثيل البيانات بشكل هرمي تفاعلي يسهل المقارنة بين الفئات المختلفة.

In [11]:
# 1. تجميع البيانات حسب فئة المنتج بالإنجليزية لحساب الإيرادات وعدد القطع
category_analysis = df_sales.groupby('product_category_name_english').agg(
    total_revenue=('price', 'sum'),
    items_sold=('order_item_id', 'count')
).reset_index()

# سنأخذ أعلى 20 فئة فقط ليكون الرسم واضحاً ومنظماً
top_categories = category_analysis.sort_values(by='total_revenue', ascending=False).head(20)

# 2. رسم الـ Treemap باستخدام Plotly Express
fig_products = px.treemap(
    top_categories, 
    path=['product_category_name_english'], # الفئات التي سيتم تقسيم المربعات بناءً عليها
    values='total_revenue', # حجم المربع يمثل الإيرادات
    color='items_sold', # لون المربع يمثل عدد القطع المباعة (تدرج لوني)
    color_continuous_scale='RdBu',
    title='أعلى 20 فئة منتجات تحقيقاً للإيرادات وعدد القطع المباعة',
    labels={
        'product_category_name_english': 'فئة المنتج',
        'total_revenue': 'إجمالي الإيرادات (BRL)',
        'items_sold': 'عدد القطع المباعة'
    }
)

# تحسين مظهر الرسم
fig_products.update_layout(title_x=0.5, template='plotly_dark')
fig_products.show()

## 💳 8. تحليل طرق الدفع وسلوك المستهلك (Payment Methods Analysis)

الآن سننتقل لجانب المعاملات المالية للإجابة على: **ما هي طريقة الدفع المفضلة لدى العملاء؟ وهل تؤثر على قيمة المشتريات؟**
سنقوم بدمج بيانات جدول المدفوعات (`order_payments`) مع جدول المبيعات ونرسم مخططاً دائرياً (Pie Chart).

In [12]:
# 1. حساب إجمالي المبيعات وعدد العمليات لكل طريقة دفع
payment_counts = order_payments.groupby('payment_type').agg(
    total_value=('payment_value', 'sum'),
    transaction_count=('order_id', 'count')
).reset_index()

# استبعاد الأنواع غير المحددة إن وجدت
payment_counts = payment_counts[payment_counts['payment_type'] != 'not_defined']

# 2. رسم Donut Chart (شكل الدونات المتطور من الـ Pie Chart) باستخدام Plotly
fig_payment = px.pie(
    payment_counts, 
    values='transaction_count', 
    names='payment_type',
    hole=0.4, # يجعلها Donut Chart بدلاً من دائرة مصمتة
    title='توزيع نسب استخدام طرق الدفع المختلفة',
    color_discrete_sequence=px.colors.sequential.Plasma_r
)

fig_payment.update_layout(title_x=0.5, template='plotly_dark')
fig_payment.show()

## 📦 9. تحليل رضا العملاء وكفاءة الشحن (Delivery & Customer Reviews)

في هذا الجزء الأخير، سنقوم بربط الجوانب التشغيلية (اللوجستية) برضا العميل من خلال:
1. **حساب وقت التوصيل الفعلي:** وهو الفرق بين تاريخ الشراء وتاريخ الاستلام الفعلي.
2. **دراسة العلاقة بين وقت التوصيل والتقييمات:** لمعرفة هل يتأثر الـ `review_score` سلباً بزيادة أيام الشحن.

سنستخدم **Plotly Express** لعمل مخطط تفاعلي (Box Plot أو Bar Chart) يوضح هذه العلاقة بوضوح.

In [13]:
# 1. حساب وقت التوصيل الفعلي بالأيام
df_sales['actual_delivery_days'] = (df_sales['order_delivered_customer_date'] - df_sales['order_purchase_timestamp']).dt.days

# 2. دمج جدول التقييمات مع الجدول الرئيسي بناءً على order_id
df_reviews = pd.merge(df_sales, order_reviews[['order_id', 'review_score']], on='order_id', how='inner')

# 3. تنظيف البيانات من القيم الفارغة في أيام التوصيل (الطلبات التي لم تصل بعد)
df_reviews_clean = df_reviews.dropna(subset=['actual_delivery_days'])

# 4. حساب متوسط أيام التوصيل لكل تقييم (من 1 لـ 5)
delivery_vs_review = df_reviews_clean.groupby('review_score')['actual_delivery_days'].mean().reset_index()

# 5. رسم Bar Chart تفاعلي يوضح تأثير أيام الشحن على التقييم
fig_reviews = px.bar(
    delivery_vs_review, 
    x='review_score', 
    y='actual_delivery_days',
    title='متوسط أيام التوصيل مقابل تقييم العملاء (Delivery Days vs Review Score)',
    labels={'review_score': 'تقييم العميل (نجوم)', 'actual_delivery_days': 'متوسط أيام التوصيل الفعلي'},
    text_auto='.1f', # إظهار الرقم بدقة رقم عشري واحد فوق كل عمود
    color='actual_delivery_days',
    color_continuous_scale='Reds' # تدرج أحمر ليدل على خطورة زيادة الأيام
)

# تحسين المظهر
fig_reviews.update_layout(title_x=0.5, template='plotly_dark')
fig_reviews.show()

## 🎯 10. الخلاصة وأهم التوصيات للبيزنس (Executive Summary & Insights)

بعد إتمام التحليل الاستكشافي الشامل باستخدام **Jupyter** و **Plotly**، وصلنا لأهم النتائج التي يمكن تقديمها لمتخذي القرار:

1. **النمو المالي:** هناك طفرة ونمو مستمر في المبيعات الشهرية، مع وجود مواسم ذروة واضحة (Seasonality) يجب الاستعداد لها بزيادة المخزون.
2. **التركيز الجغرافي:** ولاية **São Paulo (SP)** هي القوة الضاربة للموقع، حيث تستحوذ على النصيب الأكبر من العملاء والمبيعات.
3. **سلوك الدفع:** بطاقات الائتمان (**Credit Cards**) هي الوسيلة المهيمنة، ويفضل تقديم عروض تقسيط بدون فوائد لزيادة متوسط قيمة الطلب (AOV).
4. **عنق الزجاجة (الشحن والتقييمات):** التحليل أثبت بالدليل القاطع أن **تأخير الشحن هو العدو الأول لرضا العملاء**؛ حيث أن الطلبات التي حصلت على تقييم (1 نجمة) استغرقت في المتوسط وقتاً أطول بكثير في التوصيل مقارنة بالطلبات التي حصلت على (5 نجوم).

### 💡 التوصية التشغيلية:
يجب على الشركة إعادة التفاوض مع شركات الشحن في الولايات البعيدة، ووضع نظام تنبيهات مبكر للبائعين الذين يتأخرون في تسليم الشحنات لشركات النقل.

# 📊 TASK 1 — Sales & Revenue Analysis

في هذا القسم سنقوم بتحليل الأداء المالي العام للمتجر عبر الزمن، وتحديد فئات المنتجات الأكثر مبيعاً، ومعرفة كيفية تغير حجم الطلبات (Order Volume) على مدار الأشهر للإجابة على الأسئلة التالية:
1. **Which months generate the highest revenue?**
2. **Which product categories drive the most sales?**
3. **How has order volume changed over time?**

In [14]:
import plotly.express as px
import plotly.graph_objects as go

# أ. تجهيز البيانات الزمنية (حساب الإيرادات وحجم الطلبات شهرياً)
# نأخذ الطلبات المؤكدة والمكتملة فقط لضمان دقة الأرقام المالية
df_sales_delivered = df_sales[df_sales['order_status'] == 'delivered'].copy()
df_sales_delivered['year_month'] = df_sales_delivered['order_purchase_timestamp'].dt.to_period('M').astype(str)

# تجميع المبيعات وحجم الطلبات
monthly_analysis = df_sales_delivered.groupby('year_month').agg(
    total_revenue=('price', 'sum'),
    order_volume=('order_id', 'nunique') # حساب عدد الطلبات الفريدة
).reset_index()


# 1. رسم تطور إجمالي المبيعات الشهري (Revenue Trend)
fig_revenue = px.line(
    monthly_analysis, 
    x='year_month', 
    y='total_revenue',
    title='تطور إجمالي المبيعات الشهري (Monthly Revenue Trend)',
    labels={'year_month': 'الشهر والسنة', 'total_revenue': 'إجمالي المبيعات (BRL)'},
    markers=True
)
fig_revenue.update_layout(title_x=0.5, template='plotly_dark')
fig_revenue.show()


# 2. رسم تغير حجم الطلبات عبر الزمن (Order Volume Over Time)
fig_volume = px.bar(
    monthly_analysis, 
    x='year_month', 
    y='order_volume',
    title='تغير حجم الطلبات الشهري (Monthly Order Volume)',
    labels={'year_month': 'الشهر والسنة', 'order_volume': 'عدد الطلبات الفريدة'},
    text_auto=True
)
fig_volume.update_layout(title_x=0.5, template='plotly_dark')
fig_volume.show()


# 3. رسم أعلى 15 فئة منتجات قيادةً للمبيعات (Top Product Categories)
category_revenue = df_sales_delivered.groupby('product_category_name_english')['price'].sum().reset_index()
top_categories_rev = category_revenue.sort_values(by='price', ascending=False).head(15)

fig_cat_rev = px.bar(
    top_categories_rev, 
    x='price', 
    y='product_category_name_english',
    orientation='h', # شريط أفقي لسهولة القراءة
    title='أعلى 15 فئة منتجات تحقيقاً للإيرادات',
    labels={'price': 'إجمالي الإيرادات (BRL)', 'product_category_name_english': 'فئة المنتج'},
    color='price',
    color_continuous_scale='Cividis'
)
fig_cat_rev.update_layout(title_x=0.5, template='plotly_dark', yaxis={'categoryorder':'total ascending'})
fig_cat_rev.show()

# 🚚 TASK 2 — Delivery Performance Analysis

في هذا القسم، سنقوم بتحليل كفاءة وأداء عمليات الشحن والتوصيل للإجابة على الأسئلة التالية:
1. **What is the average delivery time by state?**
2. **How often are estimated delivery dates missed (Late Deliveries)?**
3. **Which regions suffer the most from delays?**

In [15]:
# أ. حساب مقاييس الشحن بالأيام
# 1. الوقت الفعلي للتوصيل (أيام)
df_sales_delivered['delivery_days'] = (df_sales_delivered['order_delivered_customer_date'] - df_sales_delivered['order_purchase_timestamp']).dt.days

# 2. حساب إذا كان الطلب متأخراً أم لا (إذا تجاوز التاريخ الفعلي التاريخ المتوقع)
df_sales_delivered['is_late'] = df_sales_delivered['order_delivered_customer_date'] > df_sales_delivered['order_estimated_delivery_date']

# ب. الإجابة على السؤال الأول والثالث: متوسط وقت التوصيل ونسبة التأخير لكل ولاية عميل
state_delivery = df_sales_delivered.groupby('customer_state').agg(
    avg_delivery_days=('delivery_days', 'mean'),
    late_rate=('is_late', 'mean'),
    total_orders=('order_id', 'nunique')
).reset_index()

# تحويل نسبة التأخير إلى مئوية لسهولة القراءة
state_delivery['late_rate_percent'] = state_delivery['late_rate'] * 100

# ترتيب الولايات تنازلياً حسب متوسط أيام التوصيل
state_delivery_sorted = state_delivery.sort_values(by='avg_delivery_days', ascending=False)


# 1. رسم متوسط أيام التوصيل لكل ولاية (Average Delivery Time by State)
fig_state_time = px.bar(
    state_delivery_sorted,
    x='customer_state',
    y='avg_delivery_days',
    title='متوسط أيام التوصيل الفعلي حسب ولاية العميل',
    labels={'customer_state': 'الولاية', 'avg_delivery_days': 'متوسط أيام التوصيل'},
    color='avg_delivery_days',
    color_continuous_scale='YlOrRd',
    text_auto='.1f'
)
fig_state_time.update_layout(title_x=0.5, template='plotly_dark')
fig_state_time.show()


# 2. رسم نسبة التوصيل المتأخر لكل ولاية (Missed Estimated Dates by State)
state_late_sorted = state_delivery.sort_values(by='late_rate_percent', ascending=False)

fig_state_late = px.bar(
    state_late_sorted,
    x='customer_state',
    y='late_rate_percent',
    title='نسبة الطلبات المتأخرة عن الوقت المتوقع حسب الولاية (%)',
    labels={'customer_state': 'الولاية', 'late_rate_percent': 'نسبة التأخير (%)'},
    color='late_rate_percent',
    color_continuous_scale='Reds',
    text_auto='.1f'
)
fig_state_late.update_layout(title_x=0.5, template='plotly_dark')
fig_state_late.show()


# 3. حساب النسبة العامة للتأخير في الموقع بالكامل للإجابة على (How often are estimated dates missed?)
overall_late_rate = df_sales_delivered['is_late'].mean() * 100
print(f"إجمالي نسبة الطلبات المتأخرة في المنصة ككل: {overall_late_rate:.2f}%")

إجمالي نسبة الطلبات المتأخرة في المنصة ككل: 7.91%


# 🏪 TASK 3 — Seller Performance Analysis

في هذا القسم، سنقوم بتقييم سلوك وأداء البائعين بناءً على حجم مبيعاتهم، ومعرفة البائعين الذين يحصلون على تقييمات منخفضة باستمرار، ودراسة ما إذا كان هناك علاقة بين سرعة التوصيل والتقييمات للإجابة على:
1. **Which sellers consistently receive low ratings?**
2. **Is there a correlation between delivery time and review score?**
3. **How does seller location affect performance?**

In [16]:
# أ. دمج جدول التقييمات مع الجدول الرئيسي للحصول على بيانات البائعين والتقييمات معاً
df_seller_reviews = pd.merge(df_sales_delivered, order_reviews[['order_id', 'review_score']], on='order_id', how='inner')

# ب. تجميع البيانات حسب البائع لحساب متوسط التقييم وحجم المبيعات ومتوسط أيام التوصيل
seller_stats = df_seller_reviews.groupby('seller_id').agg(
    avg_rating=('review_score', 'mean'),
    total_sales=('price', 'sum'),
    order_count=('order_id', 'nunique'),
    avg_delivery_time=('delivery_days', 'mean'),
    seller_state=('seller_state', 'first') # لمعرفة ولاية البائع
).reset_index()

# تصفية البائعين الذين لديهم عدد طلبات مقبول (مثلاً أكثر من 10 طلبات) لضمان دقة الحكم على التقييم المنخفض
active_sellers = seller_stats[seller_stats['order_count'] >= 10]

# 1. تحديد البائعين أصحاب التقييمات المنخفضة باستمرار (Top 10 Low Rated Sellers)
low_rated_sellers = active_sellers.sort_values(by='avg_rating', ascending=True).head(10)

fig_low_sellers = px.bar(
    low_rated_sellers,
    x='seller_id',
    y='avg_rating',
    title='أقل 10 بائعين تقييماً (الذين لديهم 10 طلبات على الأقل)',
    labels={'seller_id': 'معرف البائع (Seller ID)', 'avg_rating': 'متوسط التقييم'},
    color='avg_rating',
    color_continuous_scale='Reds_r',
    text_auto='.2f'
)
fig_low_sellers.update_layout(title_x=0.5, template='plotly_dark', xaxis_showticklabels=False) # إخفاء الـ IDs الطويلة لشكل أنظف
fig_low_sellers.show()

# 2. دراسة العلاقة بين وقت التوصيل والتقييم (Correlation between delivery time and review score)
# سنقوم برسم Scatter Plot يوضح العلاقة بين متوسط أيام توصيل البائع ومتوسط تقييمه
fig_corr = px.scatter(
    active_sellers,
    x='avg_delivery_time',
    y='avg_rating',
    size='order_count', # حجم النقطة يمثل عدد الطلبات
    color='avg_rating',
    title='العلاقة بين متوسط أيام التوصيل ومتوسط تقييم البائع',
    labels={'avg_delivery_time': 'متوسط أيام التوصيل للبائع', 'avg_rating': 'متوسط تقييم البائع'},
    trendline="ols", # إضافة خط اتجاه لإثبات العلاقة الرياضية
    color_continuous_scale='RdYlGn'
)
fig_corr.update_layout(title_x=0.5, template='plotly_dark')
fig_corr.show()

# ⭐ TASK 4 — Customer Satisfaction & Review Analysis

في هذا القسم الأخير، سنغوص في أعماق تقييمات العملاء لاستكشاف الفئات الأكثر تسبباً في عدم الرضا، ومعرفة التوزيع العام لدرجات التقييم، والإجابة على السؤال الحاسم: هل التأخير دائمًا يؤدي لتقييم سيء؟
1. **Which product categories receive the lowest ratings?**
2. **Does a late delivery always lead to a bad review?**
3. **What is the distribution of review scores overall?**

In [17]:
# 1. التوزيع العام لدرجات التقييم (Distribution of review scores overall)
review_dist = df_seller_reviews['review_score'].value_counts().reset_index()
review_dist.columns = ['review_score', 'count']

fig_pie_reviews = px.pie(
    review_dist,
    values='count',
    names='review_score',
    title='التوزيع العام لتقييمات العملاء (1-5 نجوم)',
    hole=0.4,
    color_discrete_sequence=px.colors.sequential.Viridis
)
fig_pie_reviews.update_layout(title_x=0.5, template='plotly_dark')
fig_pie_reviews.show()

# 2. فئات المنتجات التي حصلت على أقل تقييمات (Lowest Rated Product Categories)
cat_rating = df_seller_reviews.groupby('product_category_name_english').agg(
    avg_cat_rating=('review_score', 'mean'),
    order_count=('order_id', 'count')
).reset_index()

# نأخذ الفئات النشطة فقط (باعت أكثر من 50 قطعة مثلاً)
active_cats = cat_rating[cat_rating['order_count'] >= 50]
worst_cats = active_cats.sort_values(by='avg_cat_rating', ascending=True).head(10)

fig_worst_cats = px.bar(
    worst_cats,
    x='avg_cat_rating',
    y='product_category_name_english',
    orientation='h',
    title='أسوأ 10 فئات منتجات من حيث متوسط التقييم',
    labels={'avg_cat_rating': 'متوسط التقييم', 'product_category_name_english': 'فئة المنتج'},
    color='avg_cat_rating',
    color_continuous_scale='Reds'
)
fig_worst_cats.update_layout(title_x=0.5, template='plotly_dark', yaxis={'categoryorder':'total descending'})
fig_worst_cats.show()

# 3. هل التأخير دائمًا يؤدي لتقييم سيء؟ (Does a late delivery always lead to a bad review?)
# سنقارن بين نسبة التقييمات السيئة (1 و 2) والعادية/الممتازة (3 و 4 و 5) في حالة الشحنات المتأخرة
late_orders = df_seller_reviews[df_seller_reviews['is_late'] == True]
late_review_dist = late_orders['review_score'].value_counts(normalize=True).reset_index()
late_review_dist.columns = ['review_score', 'percentage']
late_review_dist['percentage'] *= 100

fig_late_effect = px.bar(
    late_review_dist,
    x='review_score',
    y='percentage',
    title='توزيع التقييمات في حالة "الطلبات المتأخرة فقط"',
    labels={'review_score': 'تقييم العميل', 'percentage': 'النسبة المئوية (%)'},
    text_auto='.1f',
    color='review_score'
)
fig_late_effect.update_layout(title_x=0.5, template='plotly_dark', showlegend=False)
fig_late_effect.show()

In [18]:
# حساب الأعمدة الإضافية التي سنحتاجها في الـ Dashboard
df_seller_reviews['delivery_days'] = (df_seller_reviews['order_delivered_customer_date'] - df_seller_reviews['order_purchase_timestamp']).dt.days
df_seller_reviews['is_late'] = df_seller_reviews['order_delivered_customer_date'] > df_seller_reviews['order_estimated_delivery_date']

# تصدير الجدول الرئيسي المدمج بالكامل كملف CSV واحد
df_seller_reviews.to_csv('olist_master_dataset_cleaned.csv', index=False)
print("تم تصدير الملف الرئيسي بنجاح وجاهز للرفع على Power BI!")

تم تصدير الملف الرئيسي بنجاح وجاهز للرفع على Power BI!


# 📑 ملخص المشروع: تحليل بيانات التجارة الإلكترونية لمنصة Olist
## (Project Executive Summary & Deliverables)

هذا المشروع يقدم تحليلاً استكشافياً شاملاً وسلوكياً (End-to-End Data Analysis) لبيانات منصة التجارة الإلكترونية البرازيلية **Olist** باستخدام **Jupyter Notebook** والرسومات البيانية التفاعلية بواسطة **Plotly**. تم تقسيم العمل إلى 4 محاور رئيسية تغطي كافة جوانب البيزنس والعمليات.

---

## 🛠️ أولاً: ماذا قدمنا في هذا المشروع؟ (What We Did)

تمت معالجة البيانات وربط الجداول (9 جداول) بناءً على مخطط العلاقات (ERD) وتنفيذ المهام التالية:

1. **TASK 1 — Sales & Revenue Analysis:** * تتبعنا تطور الأرباح الشهري وحجم الطلبات عبر الزمن (Time-Series Analysis).
   * حددنا الفئات الأكثر مبيعاً والأكثر ربحية للمنصة.
2. **TASK 2 — Delivery Performance Analysis:**
   * حسبنا الوقت الفعلي المستغرق للشحن بالأيام لكل ولاية في البرازيل.
   * قمنا بقياس نسبة الإخفاق والتأخير مقارنة بالتاريخ المتوقع الذي يظهر للعميل.
3. **TASK 3 — Seller Performance Analysis:**
   * قمنا بتقييم أداء البائعين وحجم مبيعاتهم وتحديد أصحاب التقييمات المنخفضة.
   * درسنا العلاقة الرياضية والارتباط (Correlation) بين سرعة شحن البائع وتقييمه النهائي.
4. **TASK 4 — Customer Satisfaction & Review Analysis:**
   * حللنا التوزيع العام لرضا العملاء (من 1 إلى 5 نجوم).
   * استخرجنا أسوأ فئات المنتجات تقييماً، وحللنا سلوك العميل عند حدوث تأخير في الشحن.

---

## 🎯 ثانياً: ما هي النتائج والـ Insights المكتشفة؟ (Key Findings & Outputs)

من خلال الرسومات البيانية التفاعلية التي تم بناؤها، وصلنا إلى النتائج الاستراتيجية التالية:

### 📈 1. الأداء المالي والمبيعات (Sales Insights)
* **الناتج:** هناك نمو تصاعدي مستمر في المبيعات، مع وجود مواسم ذروة واضحة (Seasonality) في نهاية العام (Black Friday).
* **الفئات القائدة:** فئات مثل `health_beauty` و `watches_gifts` و `bed_bath_table` هي المحرك الأساسي لأرباح المنصة.
* **حجم الطلبات:** زيادة الإيرادات ترافقت بشكل مباشر مع زيادة عدد الطلبات الفريدة، مما يدل على نجاح المنصة في جذب عملاء جدد.

### 🚚 2. كفاءة الشحن والعمليات (Logistics Insights)
* **المشكلة الجغرافية:** هناك تفاوت رهيب في كفاءة الشحن؛ الولايات القريبة من المركز مثل **São Paulo (SP)** يتم